Imports

In [2]:
# Calculations
import calfem.core as cfc
import numpy as np
import calfem.utils as cfu
import math
from scipy.integrate import solve_ivp
from plantml import plantml

# Mesh
import calfem.geometry as cfg
import calfem.mesh as cfm

# Visual
import calfem.vis_mpl as cfv
import matplotlib as mpl
mpl.use('TkAgg')

Could not import Matplotlib backends. Probarbly due to missing Qt.


<h1>Constants for Project</h1>

In [3]:
E = 5e9 # Young's modulus [Pa]
nu = 0.36 # Poisson's ratio [-]
alpha_exp = 60e-6 # Expansion coefficient [1/K]
rho = 549 # Density [kg/m^3]
c_p = 3600 # Specific heat capacity [J/(kg*K)]
k = 80 # Thermal conductivity [W/(m*K)]
alpha_n = 40 # Convection coefficient for top boundary [W/(m^2*K)]
alpha_c = 120 # Convection coefficient for cooling channels [W/(m^2*K)]
T_inf = 293 # Temperature of surrounding environment as well as starting temperature T_0 [W/(m^2*K)]
T_out = 285 # Temperature of hot cooling channels [W/(m^2*K)]
T_in = 277 # Temperature of cold cooling channels [W/(m^2*K)]
thickness = 1.6 # Thickness [m]

el_size_factor = 0.03 # Size factor of the elements [-]

<h2>Pre-processing</h2>

Setting up geometry

In [4]:
# Boundary markers
MARKER_T_ENV = 1                # For splines where convection occurs with the surrounding environment (boundary condition)
MARKER_T_OUT = 2                # For splines where convection occurs with the hot cooling channels (boundary condition)
MARKER_T_IN = 3                 # For splines where convection occurs with the cold cooling channels (boundary condition)
MARKER_U_DISPLACEMENT = 5       # For splines where the displacement is 0 (boundary condition)
MARKER_U_DISPLACEMENT_X =  6    # For splines where the displacement on the x-axis is 0 (symmetry condition)


g = cfg.Geometry()
#!!!### ha med i rapporten motsolsplacering

# POINTS
# corner                                            # INDEX
g.point([0.0, 0.0])                                 # 0       
g.point([0.300, 0.0])                               # 1
g.point([0.400, 0.200])                             # 2
g.point([0.0, 0.200])                               # 3

# half cooling channel
g.point([0.0, 0.100])                               # 4
g.point([0.0, 0.125])                               # 5
g.point([0.025, 0.100])                             # 6
g.point([0.0, 0.075])                               # 7

# CURVES
                                                    # INDEX
g.spline([0, 1], marker=MARKER_U_DISPLACEMENT)      # 0 (bottom)
g.spline([1, 2], marker=MARKER_U_DISPLACEMENT)      # 1 (diagonal edge right)
g.spline([2, 3], marker=MARKER_T_ENV)               # 2 (top)
g.spline([3, 5], marker=MARKER_U_DISPLACEMENT_X)    # 3 (left top straight edge)
g.circle([5, 4, 6], marker=MARKER_T_OUT)            # 4 (top quarter circle of half cooling channel)
g.circle([6, 4, 7], marker=MARKER_T_OUT)            # 5 (bottom quarter circle of half cooling channel)
g.spline([7, 0], marker=MARKER_U_DISPLACEMENT_X)    # 6 (left bottom straight edge)

# HOLES
def create_hole(center, radius, point_startid, curve_startid, marker):
    g.point(center)
    g.point([center[0], center[1] + radius])
    g.point([center[0] + radius, center[1]])
    g.point([center[0], center[1] - radius])
    g.point([center[0] - radius, center[1]])
    g.circle([point_startid + 1, point_startid, point_startid + 2], marker=marker)
    g.circle([point_startid + 2, point_startid, point_startid + 3], marker=marker)
    g.circle([point_startid + 3, point_startid, point_startid + 4], marker=marker)
    g.circle([point_startid + 4, point_startid, point_startid + 1], marker=marker)
    return [curve_startid, curve_startid + 1, curve_startid + 2, curve_startid + 3]

hole1 = create_hole([0.0875, 0.100], 0.025, 8, 7, MARKER_T_IN)
hole2 = create_hole([0.0875 * 2.0, 0.100], 0.025, 13, 11, MARKER_T_OUT)
hole3 = create_hole([0.0875 * 3.0, 0.100], 0.025, 18, 15, MARKER_T_IN)

# Generating surface
g.surface([0, 1, 2, 3, 4, 5, 6], [hole1, hole2, hole3])

# Visualising final geometry
cfv.draw_geometry(g)

Generating mesh

In [5]:
mesh = cfm.GmshMesh(g)
mesh.el_type = 2                        # Element type: Triangles
mesh.dofs_per_node = 1                  # Temperature <=> 1 degree of freedom per node
mesh.el_size_factor = el_size_factor
mesh.returnBoundaryElements = True      # To later retrieve which elements are subject to convection

coords, edof, dofs, bdofs, elementmarkers, boundaryElements = mesh.create()

cfv.figure()

# Draw the mesh.
cfv.draw_mesh(
    coords=coords,
    edof=edof,
    dofs_per_node=mesh.dofsPerNode,
    el_type=mesh.elType,
    filled=True,
    title="Battery"
        )



<h2>Solver</h2>

FEM Setup

In [6]:
ndof = len(coords)
nelem = len(edof)
[ex, ey] = cfc.coordxtr(edof,coords,dofs,3)
K=np.zeros((ndof, ndof))
f=np.zeros((ndof, 1))

Function to apply convection (add K_c to K and the )XXXXXXXXXX

In [7]:
def convection_boundary(boundaryElements, K, f, T, alpha, marker):
    for bel in boundaryElements[marker]:
        node1 = bel['node-number-list'][0]          # Node number (1-based index)
        node2 = bel['node-number-list'][1]          # Node number (1-based index)
        node1i = node1 - 1                          # Node index (0-based index)
        node2i = node2 - 1                          # Node index (0-based index)

        # The coords vector uses 0-based index, thus node1i and node2i
        L = math.sqrt((coords[node1i][0] - coords[node2i][0])**2 + (coords[node1i][1]-coords[node2i][1])**2)
        Kce = np.array([[2, 1], [1, 2]]) * alpha * L/6 * thickness
        fbc = np.array([[1/2], [1/2]]) * alpha * T * L * thickness

        # The assem function uses edof (which we directly input as the retrieved nodes). Edof is 1-based, thus node1 and node2.
        cfc.assem(np.array([node1, node2]), K, Kce, f, fbc)

Calculating stiffness matrix, adding convection and solving the final equation!

In [10]:
# The material is assumed to be isotropic, thus the consitutive matrix becomes:
D_constitutive = k*np.eye(2)

# Looping through each element, calculating the stiffness contribution using CALFEM and then assembling.
for i in range(nelem):
    ke = cfc.flw2te(ex[i,:], ey[i,:], [thickness], D_constitutive)
    cfc.assem(edof[i,:], K, ke)

# Adding convection
convection_boundary(boundaryElements, K, f, T_inf, alpha_n, MARKER_T_ENV)
convection_boundary(boundaryElements, K, f, T_out, alpha_c, MARKER_T_OUT)
convection_boundary(boundaryElements, K, f, T_in, alpha_c, MARKER_T_IN)

# We have no Dirichlet conditions but the empty arrays are nevertheless needed for cfc.solveq to not throw an exception
bc, bc_value = np.array([], 'i'), np.array([], 'f')

# Solving!
a, r = cfc.solveq(K, f, bc, bc_value)

# Drawing Temperatures (a)!
cfv.figure(fig_size=(10,10))
cfv.draw_nodal_values(a, coords, edof, dofs_per_node=mesh.dofs_per_node, el_type=mesh.el_type)
print("T_min = " + str(min(a)))
print("T_max = " + str(max(a)))
cfv.colorbar()

T_min = [[282.53696688]]
T_max = [[283.55774924]]


In [ ]:
cfv.showAndWait()